# Limpieza, análisis y estructuración de datos con Pandas (Instacart)
## Lección 2: Limpieza y estandarización (Laboratorio guiado)


In [ ]:
import pandas as pd
from pathlib import Path

# Busca data/raw desde el directorio actual o cualquiera de sus padres.
raices = (Path.cwd(), *Path.cwd().parents)
raiz_proyecto = next((ruta for ruta in raices if (ruta / 'data' / 'raw').is_dir()), None)
if raiz_proyecto is None:
    raise FileNotFoundError('No se encontró data/raw desde el directorio actual ni sus padres.')

RUTA = raiz_proyecto / 'data' / 'raw'
SALIDA = raiz_proyecto / 'data' / 'processed'
SALIDA.mkdir(parents=True, exist_ok=True)

archivos_requeridos = ('instacart_orders.csv', 'products.csv', 'order_products.csv')
faltantes = [nombre for nombre in archivos_requeridos if not (RUTA / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f'Faltan archivos en {RUTA}: {", ".join(faltantes)}')

orders_crudo = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo = pd.read_csv(RUTA / 'order_products.csv', sep=';')

registro = []
def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1_000_000, 2)
    })
    return df

def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({
        'order_id': 'int32',
        'user_id': 'int32',
        'order_number': 'int16',
        'order_dow': 'int8',
        'order_hour_of_day': 'int8',
        'days_since_prior_order': 'Int16'
    })

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (
        df['product_name'].fillna('Unknown')
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .str.title()
    )
    return df

def tipos_order_prods(df):
    return df.astype({
        'order_id': 'int32',
        'product_id': 'int32',
        'add_to_cart_order': 'Int16',
        'reordered': 'int8'
    })

orders_limpio = (
    orders_crudo.pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados).pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders).pipe(anotar, 'orders', '2 tipos')
)
products_limpio = (
    products_crudo.pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres).pipe(anotar, 'products', '1 nombres')
)
op_limpio = (
    op_crudo.pipe(anotar, 'order_products', '0 crudo')
    .pipe(tipos_order_prods).pipe(anotar, 'order_products', '1 tipos')
)

display(pd.DataFrame(registro))

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\instacart_orders.csv'

In [ ]:
# Reto 2: consultas con los DataFrames limpios

# 1. Seleccionar product_id y product_name
reto2_ej1 = products_limpio.loc[:, ['product_id', 'product_name']]
print(f'1. product_id y product_name: {reto2_ej1.shape[0]:,} filas y {reto2_ej1.shape[1]} columnas')
display(reto2_ej1.head())

# 2. Productos del departamento 1
reto2_ej2 = products_limpio.loc[products_limpio['department_id'] == 1]
print(f'2. Productos del departamento 1: {reto2_ej2.shape[0]:,} filas')
display(reto2_ej2.head())

# 3. Pedidos realizados a las 10:00
reto2_ej3 = orders_limpio.loc[orders_limpio['order_hour_of_day'] == 10]
print(f'3. Pedidos realizados a las 10:00: {reto2_ej3.shape[0]:,} filas')
display(reto2_ej3.head())

# 4. Pedidos realizados en el día 0 de la semana
reto2_ej4 = orders_limpio.loc[orders_limpio['order_dow'] == 0]
print(f'4. Pedidos con order_dow igual a 0: {reto2_ej4.shape[0]:,} filas')
display(reto2_ej4.head())

# 5. Pedidos con número de orden superior a 10
reto2_ej5 = orders_limpio.loc[orders_limpio['order_number'] > 10]
print(f'5. Pedidos con order_number > 10: {reto2_ej5.shape[0]:,} filas')
display(reto2_ej5.head())

# 6. Pedidos con más de 15 días desde el pedido anterior
reto2_ej6 = orders_limpio.loc[orders_limpio['days_since_prior_order'] > 15]
print(f'6. Pedidos con más de 15 días desde el anterior: {reto2_ej6.shape[0]:,} filas')
display(reto2_ej6.head())

# 7. Productos que son recompras
reto2_ej7 = op_limpio.loc[op_limpio['reordered'] == 1]
print(f'7. Productos recomprados (reordered = 1): {reto2_ej7.shape[0]:,} filas')
display(reto2_ej7.head())

In [ ]:
# Lección 3: perfil de hábitos de compra

# Cuenta pedidos únicos por cliente y clasifica el volumen de compra.
pedidos_por_cliente = orders_limpio.groupby('user_id')['order_id'].nunique()
segmento_por_cliente = pd.cut(
    pedidos_por_cliente,
    bins=[0, 1, 4, float('inf')],
    labels=['ocasional', 'regular', 'frecuente'],
    include_lowest=True
)

# Añade al DataFrame de pedidos el total del cliente y su segmento mediante map().
orders_limpio['n_pedidos_cliente'] = orders_limpio['user_id'].map(pedidos_por_cliente)
orders_limpio['segmento_cliente'] = orders_limpio['user_id'].map(segmento_por_cliente)

# Resume clientes, pedidos y mediana de días de espera para cada segmento.
resumen_segmento = (
    orders_limpio.groupby('segmento_cliente', observed=False)
    .agg(
        n_clientes=('user_id', 'nunique'),
        n_pedidos=('order_id', 'nunique'),
        espera_mediana=('days_since_prior_order', 'median')
    )
    .reset_index()
)
print('Resumen por segmento de cliente:')
display(resumen_segmento)

# Agrupa las 24 horas en cuatro franjas y cruza franjas con segmentos.
orders_limpio['franja_horaria'] = pd.cut(
    orders_limpio['order_hour_of_day'],
    bins=[-1, 5, 11, 17, 23],
    labels=['madrugada', 'mañana', 'tarde', 'noche'],
    include_lowest=True
)
pedidos_por_franja = pd.crosstab(
    orders_limpio['segmento_cliente'],
    orders_limpio['franja_horaria'],
    dropna=False
)
print('Pedidos por segmento y franja horaria:')
display(pedidos_por_franja)

# Conserva el segmento como atributo analítico y elimina columnas auxiliares.
orders_limpio = orders_limpio.drop(columns=['n_pedidos_cliente', 'franja_horaria'])

In [ ]:
# Reto 3: análisis del perfil de compra

# 1. Horas con más pedidos en domingo (order_dow = 0).
pedidos_domingo_por_hora = (
    orders_limpio.loc[orders_limpio['order_dow'] == 0]
    .groupby('order_hour_of_day')
    .size()
    .sort_values(ascending=False)
)
print('1. Cinco horas con más pedidos en domingo:')
display(pedidos_domingo_por_hora.head(5).rename('n_pedidos').to_frame())

# 2. Pedidos pertenecientes a clientes con más de 10 pedidos en el dataset.
pedidos_reto3_por_cliente = orders_limpio.groupby('user_id')['order_id'].nunique()
clientes_frecuentes_reto3 = pedidos_reto3_por_cliente[pedidos_reto3_por_cliente > 10].index
pedidos_clientes_frecuentes = orders_limpio.loc[
    orders_limpio['user_id'].isin(clientes_frecuentes_reto3)
]
print(
    f'2. Clientes con más de 10 pedidos: {len(clientes_frecuentes_reto3):,}; '
    f'pedidos asociados: {len(pedidos_clientes_frecuentes):,}'
)
display(pedidos_clientes_frecuentes[['order_id', 'user_id', 'order_number']].head())

# 3. Tasa de recompra según la posición en que se agregó el artículo.
recompra_por_posicion = (
    op_limpio.groupby('add_to_cart_order')
    .agg(
        n_articulos=('reordered', 'size'),
        tasa_recompra=('reordered', 'mean')
    )
)
recompra_por_posicion['tasa_recompra_pct'] = recompra_por_posicion['tasa_recompra'] * 100
print('3. Tasa de recompra por posición en el carrito:')
display(recompra_por_posicion[['n_articulos', 'tasa_recompra_pct']].head(15).round(2))

# Agrega compras y recompras por producto; se reutiliza en los ejercicios 4, 5 y 7.
metricas_producto_reto3 = op_limpio.groupby('product_id').agg(
    n_compras=('order_id', 'size'),
    n_recompras=('reordered', 'sum')
)
metricas_producto_reto3['tasa_recompra'] = (
    metricas_producto_reto3['n_recompras'] / metricas_producto_reto3['n_compras']
)

# 4. Productos con más apariciones en los carritos.
top_productos_reto3 = (
    metricas_producto_reto3.nlargest(10, 'n_compras')
    .reset_index()
    .merge(
        products_limpio[['product_id', 'product_name']],
        on='product_id',
        how='left',
        validate='one_to_one'
    )
)
print('4. Diez productos más comprados:')
display(top_productos_reto3[['product_name', 'n_compras']])

# 5. Productos con mayor tasa de recompra y al menos 100 compras observadas.
productos_con_soporte = metricas_producto_reto3.loc[
    metricas_producto_reto3['n_compras'] >= 100
]
top_tasa_recompra = (
    productos_con_soporte.nlargest(10, 'tasa_recompra')
    .reset_index()
    .merge(
        products_limpio[['product_id', 'product_name']],
        on='product_id',
        how='left',
        validate='one_to_one'
    )
)
top_tasa_recompra['tasa_recompra_pct'] = top_tasa_recompra['tasa_recompra'] * 100
print('5. Diez productos con mayor tasa de recompra (mínimo 100 compras):')
display(top_tasa_recompra[['product_name', 'n_compras', 'tasa_recompra_pct']].round(2))

# 6. Tamaño medio y mediano del carrito por día de la semana.
articulos_por_pedido_reto3 = op_limpio.groupby('order_id').size().rename('n_articulos')
tamano_carrito_por_dia = (
    orders_limpio[['order_id', 'order_dow']]
    .merge(
        articulos_por_pedido_reto3,
        left_on='order_id',
        right_index=True,
        how='inner',
        validate='one_to_one'
    )
    .groupby('order_dow')
    .agg(
        n_pedidos=('order_id', 'nunique'),
        media_articulos=('n_articulos', 'mean'),
        mediana_articulos=('n_articulos', 'median')
    )
    .reset_index()
)
print('6. Tamaño del carrito por día de la semana (0-6):')
display(tamano_carrito_por_dia.round(2))

# 7. Tasa de recompra agregada por departamento.
metricas_por_departamento = (
    metricas_producto_reto3.reset_index()
    .merge(
        products_limpio[['product_id', 'department_id']],
        on='product_id',
        how='left',
        validate='one_to_one'
    )
    .groupby('department_id')
    .agg(
        n_compras=('n_compras', 'sum'),
        n_recompras=('n_recompras', 'sum')
    )
)
metricas_por_departamento['tasa_recompra_pct'] = (
    metricas_por_departamento['n_recompras'] / metricas_por_departamento['n_compras'] * 100
)
metricas_por_departamento = metricas_por_departamento.sort_values(
    'tasa_recompra_pct', ascending=False
)
print('7. Tasa de recompra por departamento:')
display(metricas_por_departamento.round(2))